# Performance Reporting & Risk Dashboard

**CQF-Level Example**: Comprehensive performance analytics:
- Return attribution and decomposition
- Risk metrics (VaR, tracking error, information ratio)
- Drawdown analysis and recovery
- Rolling performance windows
- Client-ready reporting format

**Connectors Used:**
- `qj.eod` - Historical prices
- `qj.fred` - Risk-free rate
- `qj.ff` - Fama-French factors

**API:** https://api.quantjourney.cloud

## Run Output

![35_performance_reporting](../plots/35_performance_reporting_output_01.png)

**Prepared by QuantJourney.** Candidate notebook source is kept clean and unexecuted. Generated run artifacts are committed under `plots/` and indexed in `plots/manifest.json`.

In [ ]:
# Setup
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from scipy import stats
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "png"

from quantjourney.sdk import QuantJourneyAPI

import os
API_KEY = os.environ.get("QJ_API_KEY", "qj_...")

qj = QuantJourneyAPI(api_key=API_KEY)
print("✓ Connected to QuantJourney API")


## 1. Fetch Portfolio & Benchmark Data

In [ ]:
# Portfolio holdings
portfolio = {
    'AAPL': 0.20,
    'MSFT': 0.18,
    'GOOGL': 0.15,
    'AMZN': 0.12,
    'NVDA': 0.10,
    'META': 0.08,
    'TSLA': 0.07,
    'JPM': 0.05,
    'JNJ': 0.03,
    'V': 0.02
}

benchmark_symbol = 'SPY'
start_date = '2022-01-01'
end_date = '2024-12-31'

# Fetch prices
prices = {}
all_symbols = list(portfolio.keys()) + [benchmark_symbol]

for symbol in all_symbols:
    try:
        response = qj.eod.get_historical_prices(
            symbol=symbol,
            start_date=start_date,
            end_date=end_date,
            frequency='1d'
        )
        data = response.get('value', response) if isinstance(response, dict) else response
        if isinstance(data, list) and len(data) > 0:
            df = pd.DataFrame(data)
            df['date'] = pd.to_datetime(df['date'])
            df = df.set_index('date')
            prices[symbol] = df['adjusted_close' if 'adjusted_close' in df.columns else 'close']
            print(f"✓ {symbol}")
    except Exception as e:
        print(f"✗ {symbol}: {e}")

# Fallback
if len(prices) < 5:
    print("\nGenerating synthetic data...")
    dates = pd.date_range(start=start_date, end=end_date, freq='B')
    np.random.seed(42)
    
    for sym in all_symbols:
        mu = 0.12 + np.random.randn() * 0.05
        sigma = 0.20 + np.random.rand() * 0.15
        ret = mu/252 + sigma/np.sqrt(252) * np.random.randn(len(dates))
        prices[sym] = pd.Series(100 * np.cumprod(1 + ret), index=dates)


In [ ]:
# Build price matrix
prices_df = pd.DataFrame(prices).dropna()
returns_df = prices_df.pct_change().dropna()

# Portfolio returns
weights = pd.Series({k: v for k, v in portfolio.items() if k in returns_df.columns})
weights = weights / weights.sum()  # Normalize

portfolio_returns = (returns_df[weights.index] * weights).sum(axis=1)
benchmark_returns = returns_df[benchmark_symbol]

# Align
common_idx = portfolio_returns.index.intersection(benchmark_returns.index)
portfolio_returns = portfolio_returns.loc[common_idx]
benchmark_returns = benchmark_returns.loc[common_idx]

print(f"\nData period: {common_idx[0].date()} to {common_idx[-1].date()}")
print(f"Trading days: {len(common_idx)}")


## 2. Core Performance Metrics

In [ ]:
def calculate_performance_metrics(returns, benchmark=None, rf=0.04):
    """
    Calculate comprehensive performance metrics
    """
    metrics = {}
    
    # Basic returns
    metrics['Total Return'] = (1 + returns).prod() - 1
    metrics['CAGR'] = (1 + metrics['Total Return']) ** (252/len(returns)) - 1
    metrics['Annualized Vol'] = returns.std() * np.sqrt(252)
    
    # Risk-adjusted
    excess_return = metrics['CAGR'] - rf
    metrics['Sharpe Ratio'] = excess_return / metrics['Annualized Vol'] if metrics['Annualized Vol'] > 0 else 0
    
    # Downside risk
    downside_returns = returns[returns < 0]
    metrics['Downside Vol'] = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 0 else 0
    metrics['Sortino Ratio'] = excess_return / metrics['Downside Vol'] if metrics['Downside Vol'] > 0 else 0
    
    # Drawdown
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.cummax()
    drawdown = cum_returns / running_max - 1
    metrics['Max Drawdown'] = drawdown.min()
    metrics['Calmar Ratio'] = metrics['CAGR'] / abs(metrics['Max Drawdown']) if metrics['Max Drawdown'] != 0 else 0
    
    # Win rate
    metrics['Win Rate'] = (returns > 0).mean()
    metrics['Best Day'] = returns.max()
    metrics['Worst Day'] = returns.min()
    
    # Skewness and Kurtosis
    metrics['Skewness'] = stats.skew(returns)
    metrics['Kurtosis'] = stats.kurtosis(returns)
    
    # VaR and CVaR
    metrics['VaR (95%)'] = np.percentile(returns, 5)
    metrics['CVaR (95%)'] = returns[returns <= metrics['VaR (95%)']].mean()
    
    # Benchmark-relative (if provided)
    if benchmark is not None:
        active_returns = returns - benchmark
        metrics['Active Return'] = active_returns.mean() * 252
        metrics['Tracking Error'] = active_returns.std() * np.sqrt(252)
        metrics['Information Ratio'] = metrics['Active Return'] / metrics['Tracking Error'] if metrics['Tracking Error'] > 0 else 0
        
        # Beta
        cov = np.cov(returns, benchmark)
        metrics['Beta'] = cov[0, 1] / cov[1, 1] if cov[1, 1] != 0 else 1
        metrics['Alpha (ann)'] = metrics['CAGR'] - rf - metrics['Beta'] * (benchmark.mean() * 252 - rf)
        
        # Up/Down capture
        up_periods = benchmark > 0
        down_periods = benchmark < 0
        metrics['Upside Capture'] = returns[up_periods].mean() / benchmark[up_periods].mean() if up_periods.sum() > 0 and benchmark[up_periods].mean() != 0 else 1
        metrics['Downside Capture'] = returns[down_periods].mean() / benchmark[down_periods].mean() if down_periods.sum() > 0 and benchmark[down_periods].mean() != 0 else 1
    
    return metrics


In [ ]:
# Calculate metrics
port_metrics = calculate_performance_metrics(portfolio_returns, benchmark_returns)
bench_metrics = calculate_performance_metrics(benchmark_returns)

# Display
metrics_display = pd.DataFrame({
    'Portfolio': port_metrics,
    'Benchmark': bench_metrics
}).T

print("\n" + "="*70)
print("PERFORMANCE SUMMARY")
print("="*70)

# Format for display
pct_cols = ['Total Return', 'CAGR', 'Annualized Vol', 'Downside Vol', 'Max Drawdown', 
            'Win Rate', 'Best Day', 'Worst Day', 'VaR (95%)', 'CVaR (95%)', 
            'Active Return', 'Tracking Error', 'Alpha (ann)']

for col in metrics_display.columns:
    if col in pct_cols:
        metrics_display[col] = metrics_display[col].apply(lambda x: f"{x*100:.2f}%" if pd.notna(x) else 'N/A')
    elif col in ['Upside Capture', 'Downside Capture']:
        metrics_display[col] = metrics_display[col].apply(lambda x: f"{x*100:.1f}%" if pd.notna(x) else 'N/A')
    else:
        metrics_display[col] = metrics_display[col].apply(lambda x: f"{x:.3f}" if pd.notna(x) else 'N/A')

print(metrics_display.T.to_string())


## 3. Cumulative Performance Chart

In [ ]:
# Cumulative returns
cum_portfolio = (1 + portfolio_returns).cumprod()
cum_benchmark = (1 + benchmark_returns).cumprod()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cum_portfolio.index, y=cum_portfolio,
    mode='lines', name='Portfolio',
    line=dict(color='cyan', width=2)
))

fig.add_trace(go.Scatter(
    x=cum_benchmark.index, y=cum_benchmark,
    mode='lines', name=f'Benchmark ({benchmark_symbol})',
    line=dict(color='orange', width=2)
))

fig.update_layout(
    title='Cumulative Performance',
    xaxis_title='Date',
    yaxis_title='Growth of $1',
    template='plotly_dark',
    height=450,
    hovermode='x unified'
)
fig.show()


## 4. Drawdown Analysis

In [ ]:
def analyze_drawdowns(returns, top_n=5):
    """
    Analyze drawdown periods
    """
    cum_returns = (1 + returns).cumprod()
    running_max = cum_returns.cummax()
    drawdown = cum_returns / running_max - 1
    
    # Find drawdown periods
    in_drawdown = drawdown < 0
    dd_start = (in_drawdown & ~in_drawdown.shift(1).fillna(False))
    dd_end = (~in_drawdown & in_drawdown.shift(1).fillna(False))
    
    starts = dd_start[dd_start].index.tolist()
    ends = dd_end[dd_end].index.tolist()
    
    # Match starts with ends
    if len(ends) < len(starts):
        ends.append(returns.index[-1])
    
    dd_info = []
    for i, start in enumerate(starts[:len(ends)]):
        end = ends[i]
        period_dd = drawdown[start:end]
        if len(period_dd) > 0:
            trough_date = period_dd.idxmin()
            max_dd = period_dd.min()
            days_to_trough = (trough_date - start).days
            days_to_recovery = (end - trough_date).days if end != returns.index[-1] or drawdown.iloc[-1] >= 0 else None
            
            dd_info.append({
                'Start': start.date(),
                'Trough': trough_date.date(),
                'End': end.date() if days_to_recovery else 'Ongoing',
                'Max DD': max_dd,
                'Days to Trough': days_to_trough,
                'Days to Recovery': days_to_recovery
            })
    
    dd_df = pd.DataFrame(dd_info)
    dd_df = dd_df.sort_values('Max DD').head(top_n).reset_index(drop=True)
    
    return drawdown, dd_df

# Analyze
port_dd, port_dd_table = analyze_drawdowns(portfolio_returns)
bench_dd, bench_dd_table = analyze_drawdowns(benchmark_returns)

print("\nTop 5 Portfolio Drawdowns:")
port_dd_table['Max DD'] = port_dd_table['Max DD'].apply(lambda x: f"{x*100:.2f}%")
print(port_dd_table.to_string(index=False))


In [ ]:
# Drawdown chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=port_dd.index, y=port_dd * 100,
    fill='tozeroy', name='Portfolio',
    line=dict(color='cyan')
))

fig.add_trace(go.Scatter(
    x=bench_dd.index, y=bench_dd * 100,
    fill='tozeroy', name='Benchmark',
    line=dict(color='orange')
))

fig.update_layout(
    title='Underwater (Drawdown) Chart',
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    template='plotly_dark',
    height=400
)
fig.show()


## 5. Rolling Performance

In [ ]:
# Rolling metrics
window = 63  # ~3 months

rolling_ret = portfolio_returns.rolling(window).mean() * 252
rolling_vol = portfolio_returns.rolling(window).std() * np.sqrt(252)
rolling_sharpe = rolling_ret / rolling_vol

# Rolling tracking error
active_returns = portfolio_returns - benchmark_returns
rolling_te = active_returns.rolling(window).std() * np.sqrt(252)
rolling_ir = (active_returns.rolling(window).mean() * 252) / rolling_te

# Rolling beta
def rolling_beta(returns, benchmark, window):
    cov = returns.rolling(window).cov(benchmark)
    var = benchmark.rolling(window).var()
    return cov / var

rolling_b = rolling_beta(portfolio_returns, benchmark_returns, window)


In [ ]:
# Rolling metrics chart
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Rolling Sharpe (3M)', 'Rolling Beta (3M)', 
                    'Rolling Volatility (3M)', 'Rolling Information Ratio (3M)']
)

fig.add_trace(
    go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe, mode='lines', name='Sharpe'),
    row=1, col=1
)
fig.add_hline(y=1, line_dash="dash", line_color="green", row=1, col=1)

fig.add_trace(
    go.Scatter(x=rolling_b.index, y=rolling_b, mode='lines', name='Beta'),
    row=1, col=2
)
fig.add_hline(y=1, line_dash="dash", line_color="white", row=1, col=2)

fig.add_trace(
    go.Scatter(x=rolling_vol.index, y=rolling_vol * 100, mode='lines', name='Vol (%)'),
    row=2, col=1
)

fig.add_trace(
    go.Scatter(x=rolling_ir.index, y=rolling_ir, mode='lines', name='IR'),
    row=2, col=2
)
fig.add_hline(y=0.5, line_dash="dash", line_color="green", row=2, col=2)

fig.update_layout(
    title='Rolling Performance Metrics',
    template='plotly_dark',
    height=500,
    showlegend=False
)
fig.show()


## 6. Return Attribution

In [ ]:
# Stock-level contribution
stock_returns = returns_df[[s for s in weights.index if s in returns_df.columns]]

# Total contribution = weight * return
contributions = stock_returns * weights
total_contrib = contributions.sum()

# Attribution summary
attribution = pd.DataFrame({
    'Weight': weights,
    'Return': stock_returns.sum(),
    'Contribution': total_contrib
}).sort_values('Contribution', ascending=False)

attribution['% of Total'] = attribution['Contribution'] / attribution['Contribution'].sum() * 100

print("\nStock-Level Attribution:")
attr_display = attribution.copy()
attr_display['Weight'] = attr_display['Weight'].apply(lambda x: f"{x*100:.1f}%")
attr_display['Return'] = attr_display['Return'].apply(lambda x: f"{x*100:.1f}%")
attr_display['Contribution'] = attr_display['Contribution'].apply(lambda x: f"{x*100:.2f}%")
attr_display['% of Total'] = attr_display['% of Total'].apply(lambda x: f"{x:.1f}%")
print(attr_display.to_string())


In [ ]:
# Attribution waterfall
fig = go.Figure(go.Waterfall(
    name="Contribution",
    orientation="v",
    x=list(attribution.index) + ['Total'],
    y=list(attribution['Contribution'] * 100) + [attribution['Contribution'].sum() * 100],
    measure=['relative'] * len(attribution) + ['total'],
    textposition="outside",
    text=[f"{x*100:.1f}%" for x in attribution['Contribution']] + [f"{attribution['Contribution'].sum()*100:.1f}%"],
    connector={"line": {"color": "rgb(63, 63, 63)"}}
))

fig.update_layout(
    title='Return Attribution by Stock',
    xaxis_title='Stock',
    yaxis_title='Contribution (%)',
    template='plotly_dark',
    height=450
)
fig.show()


## 7. Calendar Returns

In [ ]:
# Monthly returns
monthly_returns = portfolio_returns.resample('M').apply(lambda x: (1+x).prod()-1)
monthly_returns.index = monthly_returns.index.to_period('M')

# Create calendar heatmap data
monthly_pivot = monthly_returns.reset_index()
monthly_pivot.columns = ['Period', 'Return']
monthly_pivot['Year'] = monthly_pivot['Period'].apply(lambda x: x.year)
monthly_pivot['Month'] = monthly_pivot['Period'].apply(lambda x: x.month)

calendar_matrix = monthly_pivot.pivot(index='Year', columns='Month', values='Return')

# Add annual returns
annual_returns = portfolio_returns.resample('Y').apply(lambda x: (1+x).prod()-1)
calendar_matrix['Annual'] = annual_returns.values[:len(calendar_matrix)]

print("\nMonthly Returns Calendar:")
cal_display = calendar_matrix.copy()
for col in cal_display.columns:
    cal_display[col] = cal_display[col].apply(lambda x: f"{x*100:.1f}%" if pd.notna(x) else '')
print(cal_display.to_string())


In [ ]:
# Calendar heatmap
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'YTD']

fig = go.Figure(data=go.Heatmap(
    z=calendar_matrix.values * 100,
    x=month_labels,
    y=calendar_matrix.index.astype(str),
    colorscale='RdYlGn',
    zmid=0,
    text=np.round(calendar_matrix.values * 100, 1),
    texttemplate='%{text}%',
    textfont={"size": 10},
    hovertemplate='%{y} %{x}: %{z:.1f}%<extra></extra>'
))

fig.update_layout(
    title='Monthly Returns Heatmap',
    xaxis_title='Month',
    yaxis_title='Year',
    template='plotly_dark',
    height=300
)
fig.show()


## 8. Risk Decomposition

In [ ]:
# Marginal contribution to risk (MCR)
cov_matrix = stock_returns.cov() * 252
portfolio_vol = np.sqrt(weights @ cov_matrix @ weights)

# MCR = (Cov * w) / sigma_p
mcr = (cov_matrix @ weights) / portfolio_vol

# Component risk = w * MCR
component_risk = weights * mcr
pct_risk = component_risk / component_risk.sum() * 100

risk_decomp = pd.DataFrame({
    'Weight': weights,
    'MCR': mcr,
    'Component Risk': component_risk,
    '% of Total Risk': pct_risk
}).sort_values('% of Total Risk', ascending=False)

print("\nRisk Decomposition:")
risk_display = risk_decomp.copy()
risk_display['Weight'] = risk_display['Weight'].apply(lambda x: f"{x*100:.1f}%")
risk_display['MCR'] = risk_display['MCR'].apply(lambda x: f"{x*100:.2f}%")
risk_display['Component Risk'] = risk_display['Component Risk'].apply(lambda x: f"{x*100:.2f}%")
risk_display['% of Total Risk'] = risk_display['% of Total Risk'].apply(lambda x: f"{x:.1f}%")
print(risk_display.to_string())


In [ ]:
# Risk vs Weight chart
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Weight',
    x=risk_decomp.index,
    y=risk_decomp['Weight'] * 100,
    marker_color='cyan'
))

fig.add_trace(go.Bar(
    name='Risk Contribution',
    x=risk_decomp.index,
    y=risk_decomp['% of Total Risk'],
    marker_color='orange'
))

fig.update_layout(
    title='Weight vs Risk Contribution',
    xaxis_title='Stock',
    yaxis_title='Percentage (%)',
    barmode='group',
    template='plotly_dark',
    height=400
)
fig.show()


## 9. Client Report Summary

In [ ]:
# Generate comprehensive report
print("="*80)
print("                    QUARTERLY PERFORMANCE REPORT")
print("="*80)
print(f"\nReporting Period: {portfolio_returns.index[0].date()} to {portfolio_returns.index[-1].date()}")
print(f"Benchmark: S&P 500 (SPY)")

print("\n" + "-"*80)
print("EXECUTIVE SUMMARY")
print("-"*80)

print(f"\n  Portfolio Total Return:    {port_metrics['Total Return']*100:>8.2f}%")
print(f"  Benchmark Total Return:    {bench_metrics['Total Return']*100:>8.2f}%")
print(f"  Excess Return:             {(port_metrics['Total Return']-bench_metrics['Total Return'])*100:>8.2f}%")

print(f"\n  Portfolio CAGR:            {port_metrics['CAGR']*100:>8.2f}%")
print(f"  Portfolio Volatility:      {port_metrics['Annualized Vol']*100:>8.2f}%")
print(f"  Sharpe Ratio:              {port_metrics['Sharpe Ratio']:>8.2f}")

print("\n" + "-"*80)
print("RISK METRICS")
print("-"*80)

print(f"\n  Maximum Drawdown:          {port_metrics['Max Drawdown']*100:>8.2f}%")
print(f"  Value at Risk (95%):       {port_metrics['VaR (95%)']*100:>8.2f}%")
print(f"  Conditional VaR (95%):     {port_metrics['CVaR (95%)']*100:>8.2f}%")
print(f"  Beta to Benchmark:         {port_metrics['Beta']:>8.2f}")
print(f"  Tracking Error:            {port_metrics['Tracking Error']*100:>8.2f}%")

print("\n" + "-"*80)
print("BENCHMARK RELATIVE")
print("-"*80)

print(f"\n  Active Return (ann):       {port_metrics['Active Return']*100:>8.2f}%")
print(f"  Information Ratio:         {port_metrics['Information Ratio']:>8.2f}")
print(f"  Upside Capture:            {port_metrics['Upside Capture']*100:>8.1f}%")
print(f"  Downside Capture:          {port_metrics['Downside Capture']*100:>8.1f}%")
print(f"  Alpha (annualized):        {port_metrics['Alpha (ann)']*100:>8.2f}%")

print("\n" + "-"*80)
print("TOP CONTRIBUTORS")
print("-"*80)

top_contributors = attribution.head(3)
for stock, row in top_contributors.iterrows():
    print(f"  {stock:>6}: {row['Contribution']*100:>+6.2f}%  (Weight: {row['Weight']*100:.1f}%)")

print("\n" + "-"*80)
print("BOTTOM CONTRIBUTORS")
print("-"*80)

bottom_contributors = attribution.tail(3).iloc[::-1]
for stock, row in bottom_contributors.iterrows():
    print(f"  {stock:>6}: {row['Contribution']*100:>+6.2f}%  (Weight: {row['Weight']*100:.1f}%)")

print("\n" + "="*80)
print("                         END OF REPORT")
print("="*80)


## 10. Dashboard Summary View

In [ ]:
# Create comprehensive dashboard
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=['Cumulative Performance', 'Monthly Returns Distribution',
                    'Drawdown', 'Risk Contribution by Stock',
                    'Rolling Sharpe (3M)', 'Weight vs Return Contribution'],
    specs=[[{}, {}], [{}, {'type': 'domain'}], [{}, {}]],  # 'domain' type for Pie chart
    vertical_spacing=0.1
)

# 1. Cumulative
fig.add_trace(
    go.Scatter(x=cum_portfolio.index, y=cum_portfolio, name='Portfolio', line=dict(color='cyan')),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=cum_benchmark.index, y=cum_benchmark, name='Benchmark', line=dict(color='orange')),
    row=1, col=1
)

# 2. Distribution
fig.add_trace(
    go.Histogram(x=portfolio_returns*100, name='Returns', marker_color='cyan', opacity=0.7),
    row=1, col=2
)

# 3. Drawdown
fig.add_trace(
    go.Scatter(x=port_dd.index, y=port_dd*100, fill='tozeroy', name='DD', line=dict(color='red')),
    row=2, col=1
)

# 4. Risk contribution pie
fig.add_trace(
    go.Pie(labels=risk_decomp.index, values=risk_decomp['% of Total Risk'], name='Risk'),
    row=2, col=2
)

# 5. Rolling Sharpe
fig.add_trace(
    go.Scatter(x=rolling_sharpe.index, y=rolling_sharpe, name='Sharpe', line=dict(color='green')),
    row=3, col=1
)
fig.add_hline(y=1, line_dash='dash', line_color='white', row=3, col=1)

# 6. Attribution
fig.add_trace(
    go.Bar(x=attribution.index, y=attribution['Contribution']*100, name='Contribution',
           marker_color=['green' if x > 0 else 'red' for x in attribution['Contribution']]),
    row=3, col=2
)

fig.update_layout(
    title='Performance Dashboard',
    template='plotly_dark',
    height=900,
    showlegend=False
)
fig.show()


## Summary

This CQF-level example covered:

1. **Performance Metrics**: CAGR, Sharpe, Sortino, Calmar ratios
2. **Drawdown Analysis**: Max drawdown, recovery periods, underwater chart
3. **Rolling Analytics**: Time-varying Sharpe, beta, tracking error
4. **Return Attribution**: Stock-level contribution analysis
5. **Calendar Returns**: Monthly performance heatmap
6. **Risk Decomposition**: Marginal and component risk contribution
7. **Benchmark Relative**: Alpha, information ratio, capture ratios
8. **Client Reporting**: Professional format quarterly report

**Family Office Applications**:
- Quarterly/annual client reporting
- Investment committee presentations
- Manager performance evaluation
- Risk monitoring and compliance